<a href="https://colab.research.google.com/github/Ersaoktaviannn/eeg-creative-state-classifier/blob/dev/State_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
import shutil

# Hapus paksa jika folder ada tapi bukan mount point yang valid
if os.path.exists('/content/drive'):
    try:
        !fusermount -u /content/drive
        shutil.rmtree('/content/drive')
    except:
        !rm -rf /content/drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import pywt
import matplotlib.pyplot as plt

from scipy.stats import shapiro, mannwhitneyu, ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore")
np.random.seed(42)

# CONFIG
DATASET_PATH = "/content/drive/MyDrive/Creativity-Dataset"
SFREQ = 500
EPOCHS = [0.25, 0.5, 1.0]
WAVELET = "db4"

# CHANNEL (63 → 16)
SELECTED_CHANNEL_INDEX = [0,1,5,10,11,15,20,21,25,26,30,31,40,41,50,51]
CHANNEL_NAMES = ["Fp1","Fp2","F7","F3","F4","F8","C3","C4","T3","T4","T5","T6","P3","P4","O1","O2"]

CREATIVE = ["IDG","IDE","IDR"]
REST = ["RST1","RST2"]

if os.path.exists(DATASET_PATH):
    files = sorted(Path(DATASET_PATH).glob("Data_Creativity_Sub_*.mat"))
    print("Total subjects found:", len(files))
else:
    print(f"Error: Folder '{DATASET_PATH}' tidak ditemukan di Drive Anda.")

Total subjects found: 28


In [3]:
def get_subject_id(name):
    return int(re.search(r"Sub_(\d+)", name).group(1))

def get_label(task):
    if task in CREATIVE: return 1
    if task in REST: return 0
    return None

def epoching(x, sf, dur):
    step = int(sf * dur)
    n = x.shape[1] // step
    return x[:, :n*step].reshape(16, n, step).transpose(1,0,2)

def hjorth(x):
    var0 = np.var(x)
    if var0 < 1e-12: return 0,0,0
    d1, d2 = np.diff(x), np.diff(np.diff(x))
    var1, var2 = np.var(d1), np.var(d2)
    mob = np.sqrt(var1/var0) if var1>0 else 0
    comp = np.sqrt(var2/var1)/mob if var1>0 and mob>0 else 0
    return var0, mob, comp

def extract_feat(epoch):
    feats = []
    for ch in epoch:
        coeffs = pywt.wavedec(ch, WAVELET, level=4)
        for c in coeffs:
            feats.extend(hjorth(c))
    return np.array(feats)

In [4]:
data = {d: {"X":[], "y":[], "s":[]} for d in EPOCHS}

for f in files:
    sid = get_subject_id(f.name)
    mat = sio.loadmat(f)

    for k in mat:
        if "__" in k: continue

        task = k.split("_")[-1]
        label = get_label(task)
        if label is None: continue

        eeg = mat[k]
        if eeg.shape[0] != 63: continue

        eeg = eeg[SELECTED_CHANNEL_INDEX]

        for d in EPOCHS:
            ep = epoching(eeg, SFREQ, d)
            if len(ep)==0: continue

            data[d]["X"].append(ep)
            data[d]["y"].append(np.full(len(ep), label))
            data[d]["s"].append(np.full(len(ep), sid))

In [5]:
dataset = {}

for d in EPOCHS:
    X = np.concatenate(data[d]["X"])
    y = np.concatenate(data[d]["y"])
    s = np.concatenate(data[d]["s"])

    # balancing
    idx_c = np.where(y==1)[0]
    idx_r = np.where(y==0)[0]
    n = min(len(idx_c), len(idx_r))

    idx = np.concatenate([idx_c[:n], idx_r[:n]])

    dataset[d] = {
        "X": X[idx],
        "y": y[idx],
        "s": s[idx]
    }

    print(f"{d}s → {len(idx)} samples")

0.25s → 68016 samples
0.5s → 34008 samples
1.0s → 17004 samples


In [6]:
features = {}

for d in EPOCHS:
    X = dataset[d]["X"]
    print(f"Extract {d}s ...")

    Xf = np.array([extract_feat(e) for e in X])

    features[d] = {
        "X": Xf,
        "y": dataset[d]["y"],
        "s": dataset[d]["s"]
    }

    print("Shape:", Xf.shape)

Extract 0.25s ...
Shape: (68016, 240)
Extract 0.5s ...


KeyboardInterrupt: 

In [23]:
def run_loso(X,y,s):
    subs = np.unique(s)
    scores = []

    for test in subs:
        tr = s!=test
        te = s==test

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr])
        Xte = scaler.transform(X[te])

        grid = GridSearchCV(
            KNeighborsClassifier(),
            {"n_neighbors":[3,5,7], "weights":["uniform","distance"]},
            cv=3, scoring="f1"
        )
        grid.fit(Xtr,y[tr])

        pred = grid.predict(Xte)

        scores.append([
            accuracy_score(y[te],pred),
            precision_score(y[te],pred),
            recall_score(y[te],pred),
            f1_score(y[te],pred)
        ])

    return np.array(scores)

Modul Feature Extraction Siap.


In [24]:
results = {}

for d in EPOCHS:
    print("\n=== Epoch", d, "===")
    sc = run_loso(features[d]["X"], features[d]["y"], features[d]["s"])

    results[d] = sc.mean(axis=0)

    print("Accuracy :", sc[:,0].mean())
    print("Precision:", sc[:,1].mean())
    print("Recall   :", sc[:,2].mean())
    print("F1       :", sc[:,3].mean())

Modul Klasifikasi Siap.


In [25]:
df = pd.DataFrame(results, index=["Acc","Prec","Rec","F1"]).T

df.plot(marker="o", figsize=(8,5))
plt.title("Model Performance vs Epoch")
plt.grid()
plt.show()

Peringatan: Tidak ada file .mat ditemukan.


In [26]:
X = features[1.0]["X"]
y = features[1.0]["y"]

pvals = []
for i in range(X.shape[1]):
    c = X[y==1,i]
    r = X[y==0,i]

    if shapiro(c[:500])[1]>0.05 and shapiro(r[:500])[1]>0.05:
        p = ttest_ind(c,r)[1]
    else:
        p = mannwhitneyu(c,r)[1]

    pvals.append(p)

print("Significant features:", np.sum(np.array(pvals)<0.05))

Modul Klasifikasi Siap.
